### Derivation of Cardano’s Method (Constructive Approach)

We begin with the general cubic:

$$
ax^3 + bx^2 + cx + d = 0
$$

---

### Step 1: Normalize

Divide by $a$:

$$
x^3 + \frac{b}{a}x^2 + \frac{c}{a}x + \frac{d}{a} = 0
$$

---

### Step 2: Completing the Cube

Starting from:

$$
x^3 + \frac{b}{a}x^2 + \frac{c}{a}x + \frac{d}{a} = 0
$$

We focus on the first two terms:

$$
x^3 + \frac{b}{a}x^2
$$

To eliminate the quadratic term, we try to express this as part of a perfect cube.

Recall:

$$
(x + k)^3 = x^3 + 3kx^2 + 3k^2x + k^3
$$

Matching coefficients:

$$
3k = \frac{b}{a} \Rightarrow k = \frac{b}{3a}
$$

This is the unique choice of $k$ that eliminates the quadratic term,
reducing the cubic to a simpler canonical form.

Now rewrite:

$$
x^3 + \frac{b}{a}x^2 = (x + \frac{b}{3a})^3 - 3\left(\frac{b}{3a}\right)^2 x - \left(\frac{b}{3a}\right)^3
$$

Substituting back into the equation:

$$
(x + \frac{b}{3a})^3 - 3\left(\frac{b}{3a}\right)^2 x - \left(\frac{b}{3a}\right)^3 + \frac{c}{a}x + \frac{d}{a} = 0
$$

Group terms:

$$
(x + \frac{b}{3a})^3 + \left(\frac{c}{a} - 3\left(\frac{b}{3a}\right)^2\right)x + \left(\frac{d}{a} - \left(\frac{b}{3a}\right)^3\right) = 0
$$

---

### Step 3: Change of Variable

Let:

$$
\mu = x + \frac{b}{3a}
\quad \Rightarrow \quad
x = \mu - \frac{b}{3a}
$$

Substituting:

$$
x = \mu - \frac{b}{3a}
$$

into the grouped equation eliminates the remaining linear term in $x$,
expresses everything in terms of $\mu$, simplifying to:

$$
\mu^3 = A\mu + B
$$

which is called the **depressed cubic**.

where:

$$
A = \frac{1}{3}\left(\frac{b}{a}\right)^2 - \frac{c}{a}
$$

$$
B = \frac{bc}{3a^2} - \frac{d}{a} - \frac{2}{27}\left(\frac{b}{a}\right)^3
$$

---

### Step 4: Key Substitution

Assume:

$$
\mu = \alpha + \beta
$$

Substitute:

$$
(\alpha + \beta)^3 = A(\alpha + \beta) + B
$$

Expanding:

$$
\alpha^3 + \beta^3 + 3\alpha\beta(\alpha + \beta) = A(\alpha + \beta) + B
$$

---

### Step 5: Enforce Constraint

To simplify, impose:

$$
3\alpha\beta = A
$$

Then:

$$
\alpha^3 + \beta^3 = B
$$

---

### Step 6: Solve for $\alpha^3, \beta^3$

Let:

$$
\alpha^3 = u, \quad \beta^3 = v
$$

Then:

$$
u + v = B
$$

$$
uv = \frac{A^3}{27}
$$

This gives a quadratic:

$$
t^2 - Bt + \frac{A^3}{27} = 0
$$

Solving:

$$
u, v = \frac{B \pm \sqrt{B^2 - \frac{4A^3}{27}}}{2}
$$

---

### Step 7: Complex Cube Roots

We now compute cube roots of $u$ and $v$.

For a complex number:

$$
z = a + ib
$$

we convert to polar form:

$$
z = r e^{i\theta}
$$

where:

$$
r = \sqrt{a^2 + b^2}, \quad \theta = \arg(z)
$$

Then cube roots are:

$$
\sqrt[3]{z} = \sqrt[3]{r} \cdot e^{i(\theta + 2k\pi)/3}, \quad k = 0,1,2
$$

This produces three possible values for each root.

---

### Step 8: Root Pairing

Not all combinations of cube roots produce valid solutions.

We enforce:

$$
\alpha \beta = \frac{A}{3}
$$

to correctly match corresponding roots.

---

### Step 9: Final Solution

The roots of the original equation are:

$$
x = \alpha + \beta - \frac{b}{3a}
$$

By cycling through valid pairs, we obtain all three roots.

---

### Key Insight

The difficulty in implementing Cardano’s method lies not in solving the equation,
but in:

- handling multi-valued complex cube roots  
- selecting correct branches  
- ensuring algebraic consistency  

These steps are essential for obtaining correct numerical solutions.

In [11]:
import mpmath as mp

mp.mp.dps = 50  # high precision

### Input Coefficients

Solve:

ax³ + bx² + cx + d = 0

Coefficients may be real or complex.

In [12]:
print("Enter coefficients a b c d (e.g. 1 0 -1 1 or 1+2j 0 -3 4j):")
a, b, c, d = [mp.mpc(complex(x)) for x in input().split()]

if abs(a) == 0:
    raise ValueError("Not a cubic equation.")

Enter coefficients a b c d (e.g. 1 0 -1 1 or 1+2j 0 -3 4j):


### Transform to Depressed Cubic

In [13]:
A = (1/3)*(b/a)**2 - c/a
B = b*c/(3*a**2) - d/a - (2/27)*(b/a)**3

### Compute Discriminant

In [14]:
D = B**2 - 4*A**3/27

### Compute Cube Roots (Polar Form)

In [15]:
alpha3 = (B + mp.sqrt(D)) / 2
beta3 = (B - mp.sqrt(D)) / 2

alpha_r, alpha_w = abs(alpha3), mp.arg(alpha3)
beta_r, beta_w = abs(beta3), mp.arg(beta3)

alpha = [
    mp.root(alpha_r, 3) * mp.e**(1j*(alpha_w + 2*k*mp.pi)/3)
    for k in range(3)
]

beta = [
    mp.root(beta_r, 3) * mp.e**(1j*(beta_w - 2*k*mp.pi)/3)
    for k in range(3)
]

### Match Correct Root Pairs

In [16]:
tol = mp.mpf('1e-10')

idx = 0
for i in range(3):
    if abs(alpha[0]*beta[i] - A/3) < tol:
        idx = i
        break

### Construct Roots

In [17]:
roots = [
    alpha[k] + beta[(k + idx) % 3] - b/(3*a)
    for k in range(3)
]

### Clean Numerical Noise

In [18]:
def clean(z, tol=mp.mpf('1e-12')):
    real = 0 if abs(mp.re(z)) < tol else mp.re(z)
    imag = 0 if abs(mp.im(z)) < tol else mp.im(z)
    return mp.mpc(real, imag)

roots = [clean(r) for r in roots]

### Output

In [19]:
print("Roots:")
for i, r in enumerate(roots, 1):
    print(f"r{i} =", mp.nstr(r, 10))

Roots:
r1 = (0.7660444431 + 0.0j)
r2 = (-0.9396926208 + 0.0j)
r3 = (0.1736481777 + 0.0j)


### Verification

In [20]:
residuals = [abs(a*r**3 + b*r**2 + c*r + d) for r in roots]

print("\nResiduals |f(r)|:")
for i, res in enumerate(residuals, 1):
    print(f"r{i}:", mp.nstr(res, 10))


Residuals |f(r)|:
r1: 5.34552942e-51
r2: 2.138211768e-50
r3: 4.009147065e-51
